# Getting the laws' discussion

With this notebook we extract the "discusion en sala" from each law

In [7]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import time
import re
from datetime import datetime, date

We start setting up the requeriments to do a proper webscrapping

In [11]:
import requests
from pathlib import Path
import re

endpoint = "https://datos.bcn.cl/sparql"
section_uri = "http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-po1-ds61"

query = f"""
PREFIX bcnres: <http://datos.bcn.cl/ontologies/bcn-resources#>
SELECT ?xml WHERE {{
  <{section_uri}> bcnres:descripcionXML ?xml .
}}
LIMIT 1
"""

r = requests.get(
    endpoint,
    params={"query": query, "format": "application/sparql-results+json"},
    headers={"Accept": "application/sparql-results+json", "User-Agent": "Mozilla/5.0"},
    timeout=30,
)
r.raise_for_status()
data = r.json()

rows = data["results"]["bindings"]
print("rows:", len(rows))

if not rows:
    print("No descripcionXML found for this section.")
else:
    xml_text = rows[0]["xml"]["value"]
    print("xml preview:", xml_text[:200])

    # VERY simple tag strip (good enough to start; you can improve later)
    plain = re.sub(r"<[^>]+>", "", xml_text)
    plain = re.sub(r"\s+\n", "\n", plain).strip()

    outdir = Path("datos_dump")
    outdir.mkdir(exist_ok=True)
    (outdir / "section_text.txt").write_text(plain, encoding="utf-8")
    print("Saved:", outdir / "section_text.txt")

rows: 1
xml preview:  <debateSection xmlns:bcn="http://datos.bcn.cl" bcn:uriProyectoLey="http://datos.bcn.cl/recurso/cl/proyecto-de-ley/16371-24" bcn:uriResultadoDebate="http://datos.bcn.cl/ontologies/bcn-resources#seApru
Saved: datos_dump\section_text.txt


In [13]:
from pathlib import Path
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

def unescape_js_string(s: str) -> str:
    # enough for BCN’s payload
    s = s.replace("\\\\", "\\")
    s = s.replace("\\'", "'")
    s = s.replace('\\"', '"')
    s = s.replace("\\n", "\n").replace("\\t", "\t").replace("\\r", "\r")
    return s

xml_path = Path("datos_dump/ley_21788_1-4.xml")  # your saved XHR response
xml_text = xml_path.read_text(encoding="utf-8")

root = ET.fromstring(xml_text)
cmd_nodes = root.findall(".//cmd[@n='js']")

target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if "div class=\"item\"" in txt and "$('.modal-body').html(" in txt:
        target_js = txt
        break

if target_js is None:
    raise ValueError("Could not find injected transcript HTML in the XAJAX XML.")

start_token = "$('.modal-body').html('"
start = target_js.find(start_token) + len(start_token)
end = target_js.rfind("');")
raw_html = target_js[start:end]

modal_html = unescape_js_string(raw_html)
soup = BeautifulSoup(modal_html, "html.parser")
item = soup.select_one("div.item")

section_uri = item.get("parteDocumento")   # <-- THIS is what you need
doc_uri = item.get("uriDocumento")

print("section_uri:", section_uri)
print("doc_uri:", doc_uri)

ParseError: no element found: line 1, column 0 (<string>)